# 第7课：高级特性 — 生成器、装饰器与并发

> **学习目标**：掌握 Python 的独门武器：生成器、装饰器、并发入门

---

## 生成器 Generator：按需生产，极致省内存

### 先理解问题

当你写 `[x**2 for x in range(1000000)]` 时，Python 会**一次性生成 100 万个元素**，全部塞进内存。如果数据量再大（比如 10GB 的日志文件），内存就不够用了。

### 生成器的解决方案

生成器不一次性生成所有数据，而是**你什么时候要，它就什么时候产一个**。用 `yield` 替代 `return`——`yield` 会"暂停"函数，把值交出去，等你下次要的时候再继续执行。

就像读书时用书签标记进度。生成器就是被 `yield` 打了"书签"的函数。每次 `next()` 就读到下一个书签处。

### 生成器为什么能"记住"状态？

生成器能记住执行位置，靠的是 CPython 底层的**帧对象（frame object）**。每个函数调用都对应一个栈帧，帧里有三个关键东西：程序计数器 `f_lasti`（当前执行到哪个字节码指令）、局部变量字典 `f_locals`（变量值快照）、行号 `f_lineno`。普通函数 return 后帧直接销毁，但生成器执行到 `yield` 时——帧不销毁，而是整个"挂起"：`f_lasti` 冻结在 `YIELD_VALUE` 指令处，`f_locals` 保留所有变量的当前值，帧从调用栈上摘下。下次 `next(gen)` 时，帧重新接回调用栈，从 `f_lasti + 1` 恢复执行，变量值原封不动回到暂停前的那一刻。

### for 循环在背后如何驱动生成器？

每次你写 `for x in gen:`，Python 解释器都悄悄展开成这样：

```python
_iter = iter(gen)
while True:
    try:
        x = next(_iter)   # 帧恢复 → 推进 → 再次 yield → 帧挂起 → 返回
    except StopIteration: # 生成器耗尽，自动跳出
        break
```

每次 `next(gen)` 都触发完整一轮"帧恢复→推进→yield→帧挂起→返回"。生成器是惰性的——不 `next()` 就不动，绝不白费 CPU。

In [ ]:
import sys

# ================================================================
# 列表 vs 生成器的内存对比
# ================================================================

# 列表推导式 [x**2 for x in ...] 使用方括号，会一次性计算出所有平方值并
# 全部存入内存中的一个列表对象。当 range(1000000) 时，列表中将包含
# 一百万个整数，每个整数在 Python 中大约占用 28 字节，合计约 28 MB。
nums_list = [x**2 for x in range(1000000)]  # 方括号 = 列表

# 生成器表达式 (x**2 for x in ...) 使用圆括号，它不会计算任何元素。
# 它只返回一个生成器对象——一个"惰性求值"的迭代器。生成器本身只保存
# 当前执行状态（帧指针、局部变量），而不保存任何计算结果。
# 这就是"惰性求值"：只在被要求时才计算下一个值。
nums_gen = (x**2 for x in range(1000000))   # 圆括号 = 生成器

# sys.getsizeof() 返回对象本身占用的内存，不递归计算内部元素占用的全部内存。
# 对列表而言，这只反映列表对象（含指针数组）的大小，不反映所有 int 对象。
# 对生成器而言，这真实反映了生成器对象的固定开销（约 200 字节），
# 因为生成器内部不存储任何计算结果。
print(f"列表内存: {sys.getsizeof(nums_list):,} 字节")
print(f"生成器内存: {sys.getsizeof(nums_gen):,} 字节")

# 生成器是"懒惰"的 — 你不问它要，它就不产。
# next(nums_gen) 从生成器暂停的位置恢复执行，计算出下一个值后再次暂停。
# 每次 next() 调用都触发：从暂停帧恢复 → 执行到下一个 yield → 返回 → 帧挂起。
# 这里用列表推导式一次性取出前 5 个值——但注意 nums_gen 已经被"消耗"了 5 个元素。
print("前5个:", [next(nums_gen) for _ in range(5)])

# 生成器只能遍历一次！已经消耗 5 个了。再次遍历将从第 6 个元素开始，
# 而不是从头开始。这与列表不同——列表可以反复多次遍历。
# 如果需要从头遍历，必须重新创建生成器对象。

### yield vs return：两种"交出值"的本质差异

- **return**：函数结束，值返回，**局部状态销毁**。就像一本书看完了合上，再打开只能从第一页看起
- **yield**：函数**暂停**，值返回，**局部状态保留**。下次 `next()` 从暂停处继续。就像书签

这就是生成器能"记住"自己执行到哪里的秘密。每个 `yield` 就是一个检查点。

#### 帧层面的对比

| 对比维度 | return | yield |
|---------|--------|-------|
| 帧的命运 | 销毁，局部变量释放 | 挂起，`f_lasti` 和 `f_locals` 完整保留 |
| 能否恢复 | 不能，再次调用创建新帧 | 能，帧重新接回调用栈 |
| 执行路径 | 函数开头重来 | 从 `f_lasti + 1` 继续 |
| 调用多少次 | 每次都独立创建新帧 | 共享同一个帧的生命周期 |

#### yield 是双向通道，不只是产出值

yield 在生成器内部是一个**表达式**，它本身也有值——值是什么取决于调用方怎么驱动它：

```python
def interactive():
    val = yield "给我点东西"
    print(f"收到了: {val}")
```

| 驱动方式 | yield 表达式的值 | 用途 |
|---------|------------------|------|
| `next(gen)` | `None` | 单向消费数据 |
| `gen.send(value)` | `value` | 双向通信（协程雏形） |
| `gen.throw(exc)` | 异常在 yield 处抛出 | 让生成器内部处理异常 |
| `gen.close()` | `GeneratorExit` 被抛出 | 安全终止生成器 |

这个双向通信机制是 Python 协程的基石，`async`/`await` 语法就是在 yield 协议上演进出来的。

In [ ]:
# ================================================================
# 生成器函数：yield 与 next() 的交互
# ================================================================

# 定义生成器函数。注意：调用 countdown() 不会立即执行函数体，
# 而是返回一个生成器对象。函数体中的代码将在第一次调用 next() 时才开始执行。
def countdown(n):
    # 函数体直到第一次 next() 调用时才真正执行
    print("开始倒计时！")
    while n > 0:
        yield n          # 暂停，交出n，保留状态
        # 解释器层面：yield 语句将生成器帧（frame）挂起，把表达式的值
        # 返回给调用者。生成器帧的 f_lasti 字段更新为 yield 指令的偏移量，
        # 局部变量（n）保存在帧的 f_locals 中。下次调用 next() 时，
        # 解释器从该帧的 f_lasti+1 处恢复执行，就像从未离开过。
        n -= 1
    # 当 while 条件不满足时，函数自然结束→抛出 StopIteration 异常
    print("倒计时结束！")

gen = countdown(3)

# 首次调用 next(gen)：生成器帧被创建，函数体开始执行，执行到 yield n 处暂停，
# 返回 n 的值（3）。生成器帧被挂起，等待下一次 next()。
print(next(gen))   # "开始倒计时！" → 交出3 → 暂停

# 再次调用 next(gen)：从上次暂停的 yield 语句之后恢复执行，执行 n -= 1，
# 回到 while 开头判断 n > 0，再次遇到 yield n，返回 n 的值（2）后再次暂停。
print(next(gen))   # 继续 → 交出2 → 暂停

# 第三次调用 next(gen)：同样的流程，返回 1 后暂停。
print(next(gen))   # 继续 → 交出1 → 暂停

# 第四次调用 next(gen)：此时 n = 0，while 条件不成立，循环结束。
# 函数自然返回 → Python 抛出 StopIteration 异常，表示生成器已耗尽。
# 如果解除以下注释，将看到 StopIteration：
# next(gen)  # StopIteration

# ================================================================
# 生成器的最大价值：表达无限序列
# ================================================================

# 列表无法表示无限序列——列表推导式会无限循环并耗尽内存。
# 生成器通过惰性求值天然支持无限序列：只在被要求时计算下一个值。
def fibonacci():
    a, b = 0, 1
    while True:          # 无限循环！列表做不到
        # yield b 将当前斐波那契数值返回给调用者，然后函数暂停。
        # 下一次 next() 从 yield 之后恢复，计算下一对 a, b。
        # 也就是说：生成器每步只产出一个值，但"记得"自己算到了哪里。
        yield b
        a, b = b, a + b  # 并行赋值：先计算右侧的 (b, a+b)，再赋给 (a, b)

fib = fibonacci()
print("\n前10个斐波那契数:", [next(fib) for _ in range(10)])
# 取完前 10 个后，fib 生成器仍然"活着"，内部 a, b 保存着第 10 对值。
# 可以继续调用 next(fib) 获取更多——理论上可以无限取下去。

---

## 装饰器 Decorator：不改源码，添加功能

### 问题场景

你想知道每个函数执行了多久。不用装饰器：每个函数里都加计时代码 → 重复、侵入、易漏。如果是第三方库的函数，你根本改不了源码。

### 装饰器的解决方案

装饰器是一个函数，**接收一个函数，返回一个新函数**。新函数 = 原功能 + 额外功能。

#### @ 语法糖到底做了什么？

`@timer` 看起来神秘，其实等价于一行代码：

```python
@timer
def slow():
    ...

# ↑ 完全等价于 ↓

def slow():
    ...
slow = timer(slow)    # 在函数定义后立即执行，把原函数替换为 wrapper
```

**执行时机很重要**：装饰器在**函数定义时**（而非调用时）就执行了。当你 `import` 这个模块时，`timer(slow)` 已经被调用，返回的 `wrapper` 绑定到 `slow` 这个名字。此后每次调用 `slow()`，实际调用的是 `wrapper()`，而 `wrapper` 内部持有原始的 `slow`。

#### 装饰器就是闭包的经典应用

`timer` 函数内部定义了 `wrapper`，`wrapper` 引用了外层的 `func`——这就形成了**闭包**。即使 `timer(slow)` 已经返回，`wrapper` 仍然"记得" `func` 指向原始函数。这是因为 `func` 作为自由变量被捕获到了 `wrapper.__closure__` 中。

#### 为什么 functools.wraps 必不可少？

没有 `functools.wraps`，装饰后的函数会"身份丢失"：

```python
@timer
def slow(): "这个函数很慢"; ...

print(slow.__name__)   # 没有 wraps → "wrapper"，有 wraps → "slow"
print(slow.__doc__)    # 没有 wraps → None，有 wraps → "这个函数很慢"
```

`functools.wraps` 将原函数的 `__name__`、`__doc__`、`__module__`、`__qualname__`、`__annotations__` 等元数据复制到 `wrapper` 上。这关系到调试（报错栈显示正确的函数名）、文档生成（`help()` 显示正确文档）、以及依赖函数元数据的框架（`pytest`、`unittest.mock` 等）。**给自己写的每个装饰器都加上 `@functools.wraps`，这是最佳实践。**

In [ ]:
import time
import functools

# ================================================================
# 装饰器：不修改原函数，在其外层包裹额外功能
# ================================================================

def timer(func):
    """
    计时装饰器。
    装饰器接收一个函数 func 作为参数，返回一个新函数 wrapper。
    新函数 wrapper 在执行原函数前后插入计时逻辑。
    这是一个典型的"闭包"：wrapper 捕获了外部变量 func。
    """

    # @functools.wraps(func) 是包裹 wrapper 的装饰器，它将原函数 func 的
    # 元数据（__name__、__doc__、__module__、__dict__、__qualname__、
    # __annotations__ 等）复制到 wrapper 函数上。
    # 为什么重要？
    #   1. 调试时看到的是原函数名而不是 "wrapper"
    #   2. 文档字符串（docstring）能正常显示
    #   3. 使用 help() 查看函数时呈现原始签名和文档
    #   4. 一些依赖函数元数据的工具（如 unittest.mock、pytest）能正常工作
    # 没有 @functools.wraps，所有被 timer 装饰的函数都会显示为 wrapper，
    # 导致调试困难、文档丢失。
    @functools.wraps(func)  # 保留原函数的名称和文档
    def wrapper(*args, **kwargs):
        # wrapper 是装饰器返回的新函数。
        # 它通过闭包记住了外层 timer 的 func 参数——func 被捕获到
        # wrapper 的 __closure__ 属性中，只要 wrapper 存活，func 就不会被 GC 回收。
        start = time.time()
        # 调用原函数 func，传入所有位置参数和关键字参数。
        # 这行是装饰器模式的核心：在"额外功能"的包围中执行原函数。
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"[计时] {func.__name__}() 耗时 {elapsed:.4f}s")
        return result
    return wrapper

# @timer 是装饰器语法糖，等价于：slow = timer(slow)
# 在函数定义时立即执行：timer(slow_func) → 返回 wrapper → 绑定到名称 slow。
# 此后调用 slow() 时，实际调用的是 wrapper()，wrapper 内部再调用原始的 slow()。
@timer
def slow():
    time.sleep(1.5)
    return "慢任务完成"

@timer
def fast():
    return "快任务完成"

print(slow())
print(fast())

# @timer 等价于: slow = timer(slow)
# 函数定义时就把"计时衣"穿上了

### 带参数的装饰器：装饰器工厂模式

有时装饰器本身需要参数（如重试次数、超时时间）。那再加一层就好：

```python
@retry(max_attempts=3, delay=1)  # 先调 retry(3,1) → 返回 decorator
def func():                       # decorator 再接收 func → 返回 wrapper
    ...
```

#### 三层嵌套的闭包链

```
retry(max_attempts=3, delay=1)           ← 第1层：装饰器工厂
  ↓ 返回 decorator，它捕获了 max_attempts 和 delay

decorator(unstable_call)                 ← 第2层：真正的装饰器
  ↓ 返回 wrapper，它捕获了 func + 来自第1层的 max_attempts 和 delay

wrapper(*args, **kwargs)                 ← 第3层：包装函数
```

闭包链传递变量：`wrapper.__closure__` 包含 `func`，`decorator.__closure__` 包含 `max_attempts` 和 `delay`。只要 wrapper 存活，沿着整条闭包链的所有变量都不会被 GC 回收。这就是 Python 的函数式基础——闭包让"函数返回函数"成为可能。

**真实场景**：重试装饰器最适合调用不可靠的外部 API（网络波动、限流）。同样的工厂模式可以轻松实现超时装饰器、缓存装饰器、权限校验装饰器等。参数化让装饰器更灵活，不需要为每个变体写一个独立的装饰器。

In [ ]:
# ================================================================
# 带参数的装饰器：装饰器工厂模式
# ================================================================
#
# 装饰器三层嵌套结构：
#   1. retry(max_attempts, delay)        ← 外层：装饰器工厂
#        ↓ 返回
#   2. decorator(func)                   ← 中层：真正的装饰器
#        ↓ 返回
#   3. wrapper(*args, **kwargs)          ← 内层：包装函数（闭包）
#
# 调用流程：@retry(max_attempts=3, delay=1)
#   第1步：retry(3, 1) 被调用 → 返回 decorator（它记住了 max_attempts 和 delay）
#   第2步：@decorator 等价于 unstable_call = decorator(unstable_call)
#         decorator(unstable_call) 接收原函数 → 返回 wrapper
#         wrapper 通过闭包同时记住了 decorator 的 max_attempts/delay
#         和 retry 的 func（经过两层闭包链：wrapper → decorator → retry）
#   第3步：调用 unstable_call() 时实际执行 wrapper()

def retry(max_attempts=3, delay=1):
    """
    第1层：装饰器工厂函数。
    接收装饰器的参数，返回真正的装饰器。
    max_attempts 和 delay 被中层 decorator 的闭包捕获。
    """

    def decorator(func):
        """
        第2层：真正的装饰器。
        接收被装饰的函数 func，返回包装函数 wrapper。
        func、max_attempts、delay 全部被内层 wrapper 的闭包捕获。
        """

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            """
            第3层：包装函数。
            闭包捕获了：
            - func（来自 decorator 参数）
            - max_attempts、delay（来自 retry 参数，经由 decorator 传递）
            - 多层嵌套的闭包链：wrapper 的 __closure__ 包含 func，
              decorator 的 __closure__ 包含 max_attempts 和 delay。
              只要 wrapper 对象存活，沿着闭包链的所有变量都不会被 GC 回收。
            """
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts:
                        # 最后一次失败：不再重试，直接向上抛出异常
                        raise
                    print(f"  第{attempt}次失败: {e}，重试...")
                    time.sleep(delay)
        return wrapper
    return decorator

import random

@retry(max_attempts=3, delay=0.3)
def unstable_call():
    if random.random() < 0.6:
        raise ConnectionError("网络超时")
    return "请求成功！"

try:
    print(unstable_call())
except ConnectionError:
    print("3次全部失败")

### 闭包：装饰器的基石

**闭包 = 函数 + 它记住的外部变量。** 当一个嵌套函数引用了外部函数的变量，就形成了闭包。关键特性：即使外部函数已返回，这些变量仍然存活。

#### 闭包的底层实现：cell 对象

Python 函数内部有一个 `__closure__` 属性，它是一个由 `cell` 对象组成的元组。每个被捕获的外部变量对应一个 `cell` 对象：

- **编译时**：Python 编译器检测到内部函数引用了外部变量，将这些变量标记为**自由变量（free variable）**，存入 `__code__.co_freevars` 元组
- **运行时**：`__closure__` 中的第 i 个 cell 对应 `co_freevars` 中的第 i 个变量名
- `cell.cell_contents`：存储变量的当前值。只要闭包函数存活，cell 对象就存在，被捕获的变量就不会被 GC 回收

```python
def make_multiplier(n):
    def multiply(x):
        return x * n        # n 被闭包捕获
    return multiply

double = make_multiplier(2)
print(double.__code__.co_freevars)          # ('n',)
print(double.__closure__[0].cell_contents)  # 2
```

`make_multiplier(2)` 返回后其帧已被销毁，但 `n=2` 这个值通过 `double.__closure__[0].cell_contents` 仍然存活。

#### 闭包捕获的是变量，不是值——经典陷阱

这是 Python 闭包最容易出错的地方——闭包捕获的是**变量本身**（引用），而不是变量在闭包创建时的值：

```python
funcs = []
for i in range(3):
    funcs.append(lambda: i)   # 所有 lambda 捕获的都是同一个变量 i

print([f() for f in funcs])   # [2, 2, 2]，不是 [0, 1, 2]！
```

为什么？循环中 `i` 是同一个变量，三个 lambda 共享这个 `i`。循环结束后 `i = 2`，所以三个函数都返回 2。解决方案是用默认参数来"冻结"当前值：`lambda x=i: x`。默认参数在**定义时求值**，将 `i` 的当前值复制到 `x` 的默认值中，每个 lambda 得到独立的 `x`。

#### 什么时候形成闭包？什么时候不？

形成闭包需要三个条件：
1. 存在**嵌套函数**（函数内部定义函数）
2. 内部函数**引用了外部函数的变量**
3. 外部函数**将内部函数返回**（或传递到外部）

如果内部函数只用自己的参数或局部变量、没有引用外部变量，就不会形成闭包，`__closure__` 为 `None`。

理解闭包的 cell 机制，你就知道装饰器的 `wrapper` 为什么能"记住"原函数——`func` 参数活在了 `wrapper.__closure__` 中。

In [ ]:
# ================================================================
# 闭包：函数 + 它"记住"的外部变量
# ================================================================
#
# 闭包的定义：当一个嵌套函数引用了其外部函数的变量时，就形成了闭包。
# 关键特性：即使外部函数已经返回，其内部的局部变量仍然被嵌套函数持有，
# 不会随外部函数结束而被销毁。
#
# 实现机制：
#   - Python 函数对象有一个 __closure__ 属性（tuple of cell objects）
#   - 每个被捕获的变量对应一个 cell object
#   - cell.cell_contents 存储变量的当前值
#   - 只要闭包函数对象存活，这些 cell 对象就不会被 GC 回收
#
# 闭包变量的生命周期：
#   创建：当外部函数被调用、嵌套函数被定义时，Python 编译器检测到
#        内部函数引用了外部变量，会将这些变量标记为"自由变量"（free variable），
#        存储在函数对象的 __code__.co_freevars 中。
#   存活：从闭包函数对象创建开始，持续到闭包函数对象被 GC 回收为止。
#   死亡：当没有任何引用指向闭包函数对象时，该对象被 GC 回收，
#        同时回收 __closure__ 中所有 cell 对象，被捕获的变量也随之释放。

def make_multiplier(n):
    """
    外部函数 make_multiplier 接收 n 作为参数。
    Python 编译器在看到内部函数 multiply 中引用了 n 时，
    将 n 标记为自由变量（free variable）。
    """

    def multiply(x):
        # n 被闭包"捕获"：multiply 的 __code__.co_freevars 包含 ('n',)
        # multiply 的 __closure__[0].cell_contents 在运行时保存 n 的值
        return x * n    # n 被闭包"捕获"
    return multiply

double = make_multiplier(2)   # multiply 记住 n=2
triple = make_multiplier(3)   # multiply 记住 n=3

# 此时 make_multiplier(2) 已经返回并结束，但 double 依然持有 n=2。
# make_multiplier(2) 调用的栈帧已被销毁，但 n=2 这个值通过
# double.__closure__[0].cell_contents 仍然存活。
print(double(10), triple(10))  # 20 30

# ================================================================
# 实用例子：日志函数工厂
# ================================================================

# 闭包的经典应用：函数工厂。make_logger 根据 level 参数
# 生成不同的 log 函数。每个生成的 log 函数都"记住"了自己的 level。
def make_logger(level):
    def log(msg):
        # level 被闭包捕获。注意：每次调用 make_logger 时，
        # level 都是该次调用的独立变量，互不干扰。
        print(f"[{level}] {msg}")
    return log

info = make_logger("INFO")
error = make_logger("ERROR")

# info 的 __closure__[0].cell_contents == "INFO"
# error 的 __closure__[0].cell_contents == "ERROR"
# 虽然两个闭包来自同一个工厂函数，但它们捕获的是各自独立的 level 变量。

info("服务启动")     # [INFO] 服务启动
error("连接失败")    # [ERROR] 连接失败

---

## 上下文管理器进阶

### with 语句的底层协议

每次你写 `with obj as x:`，Python 实际上调用的是：

1. **进入 with 块**：调用 `obj.__enter__()`，返回值赋给 `x`
2. **执行 with 块**：运行块内的代码
3. **正常退出**：调用 `obj.__exit__(None, None, None)`——三个 None 表示没有异常
4. **异常退出**：调用 `obj.__exit__(exc_type, exc_val, exc_tb)`——传入异常信息。如果 `__exit__` 返回 `True`，异常被"吞掉"（不向外传播）；返回 `False` 则异常继续传播

这就是上下文管理器的完整协议——**enter/exit 两端的钩子函数**。内置类型如 `open()` 返回的文件对象、`threading.Lock` 都实现了这个协议，但你也需要自定义的场景——比如计时、事务、资源池。

### @contextmanager：用生成器语法包装上下文管理器

`@contextmanager` 把一个生成器函数包装成实现了 `__enter__` 和 `__exit__` 的上下文管理器类。它内部做的：

1. `__enter__` 时：调用生成器函数 → 得到生成器对象 → 调用 `next(gen)` 驱动到 `yield` → 返回 yield 的值作为 `as` 子句的值
2. `__exit__` 时（无异常）：调用 `next(gen)` 让生成器从 yield 后继续执行清理代码
3. `__exit__` 时（有异常）：调用 `gen.throw(exc)` 在 yield 处注入异常。生成器内部可以选择处理。如果 `__exit__` 返回 True，异常被抑制

```python
@contextmanager
def my_context():
    # ← 这部分在 __enter__ 中执行
    setup()
    yield value   # ← __enter__ 的返回值
    # ← 这部分在 __exit__ 中执行（即使 with 块内抛异常）
    cleanup()
```

这就是"yield 之上的代码进入时执行，yield 之下的代码退出时执行（即使发生异常）"这句话的底层含义。`@contextmanager` 帮我们省略了手写 `class.__enter__`/`__exit__` 的样板代码。

In [ ]:
from contextlib import contextmanager
import time

# ================================================================
# @contextmanager：用生成器语法创建上下文管理器
# ================================================================
#
# @contextmanager 是一个装饰器，它接收一个生成器函数，将其转化为
# 上下文管理器类（实现了 __enter__ 和 __exit__ 方法）。
#
# 工作原理：
#   1. with timer("数据导出"): 执行时，contextmanager 调用生成器函数，
#      得到一个生成器对象，然后调用 next(gen) 驱动生成器执行到第一个 yield。
#   2. yield 之前的代码相当于 __enter__ 方法。
#   3. yield 语句本身相当于 __enter__ 返回给 with ... as 子句的值。
#   4. 然后 with 块内的代码开始执行。
#   5. with 块结束后（正常或异常），contextmanager 调用 gen.throw() 或
#      next(gen) 驱动生成器从 yield 后继续执行。
#   6. yield 之后的代码相当于 __exit__ 方法，即使发生异常也会执行清理。

@contextmanager
def timer(name=""):
    # with 块进入时执行的部分（等价于 __enter__）
    print(f"{name} 开始...")
    start = time.time()

    # yield 是分界线：
    #  - yield 之上：进入 with 时自动执行
    #  - yield 处：生成器暂停，控制权交给 with 块
    #  - yield 之下：退出 with 时自动执行（即使 with 块内发生异常）
    yield              # ← with块内的代码在这里执行

    # with 块退出时执行的部分（等价于 __exit__）
    # 即使 with 块内抛出异常，这行代码也会执行。
    # contextmanager 装饰器内部会捕获异常，驱动生成器执行到这里，
    # 如果异常发生但本部分没有再次抛出，则异常被"吞掉"。
    elapsed = time.time() - start
    print(f"{name} 结束，耗时 {elapsed:.2f}s")

with timer("数据导出"):
    time.sleep(0.5)
    print("  导出中...")

# 即使 with 块内抛异常，yield 之后的清理代码也会执行

---

## 并发编程入门

### 为什么需要并发？

程序经常在"等待"而非"计算"——等网络、等文件、等数据库。串行的话大部分时间在空等。**并发让程序在等的时候去干别的事。**

### GIL 的真相：Python 并发的根本约束

**GIL（Global Interpreter Lock，全局解释器锁）** 是 CPython 设计中的核心约束：同一时刻，**只有一个线程能执行 Python 字节码**。每执行一段字节码（默认每 100 个内部 ticks，或遇到 IO 阻塞时），当前线程释放 GIL，其他线程竞争获取。

这意味着：

| 任务类型 | 多线程效果 | 原因 |
|---------|-----------|------|
| **CPU 密集型**（数学计算、图像处理） | **负优化**，比串行更慢 | 线程竞争 GIL 产生上下文切换开销 |
| **IO 密集型**（网络请求、文件读写） | **有效提升**吞吐 | 线程在系统调用时释放 GIL，其他线程趁机执行 |
| **混合型** | 部分提升 | IO 等待期间的计算可以被其他线程利用 |

如果任务是纯计算的，多线程不仅不加速反而减速。要真正突破 GIL 做 CPU 并行：使用 `multiprocessing`（每个进程有独立 GIL）或写 C 扩展。

### Python 三种并发模型的选择指南

| 模型 | 适用场景 | 开销 | 共享内存 | 推荐工具 |
|------|---------|------|---------|---------|
| **线程（threading）** | 少量 IO 任务（<100 连接） | 中等（MB 级/线程） | 容易共享，需加锁 | `ThreadPoolExecutor` |
| **进程（multiprocessing）** | CPU 密集计算 | 高（GB 级/进程） | 不能直接共享，需 IPC | `ProcessPoolExecutor` |
| **协程（asyncio）** | 大量 IO 任务（上千连接） | 极低（KB 级/协程） | 不共享（单线程无需锁） | `asyncio` |

### 实战决策速查

| 场景 | 选什么 | 理由 |
|------|--------|------|
| 下载 10 个网页 | `ThreadPoolExecutor` | 简单直观，够用 |
| 处理 10000 个 WebSocket 连接 | `asyncio` | 协程开销极低，上万连接轻松扛 |
| 渲染 100 帧 4K 图像 | `ProcessPoolExecutor` | 纯 CPU 计算，必须绕过 GIL |
| 同时做以上所有事 | 组合使用 | 不同任务用不同模型 |

**一句话速记**：等 IO 的时候用线程或协程（前者方便后者高性能），算 CPU 的时候用多进程。单就一个简单脚本来说——先用 `ThreadPoolExecutor` 最稳妥，真遇到瓶颈再换协程或多进程。

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time, random

# ================================================================
# ThreadPoolExecutor 实现 IO 密集型并发
# ================================================================

def download(url):
    delay = random.uniform(0.5, 1.5)
    # time.sleep() 是 IO 阻塞操作。在 CPython 中，当线程执行到
    # time.sleep() 或其他系统调用（如 read(), write(), select()）时，
    # 会释放 GIL（全局解释器锁），让其他等待中的线程获得 GIL 并执行。
    # 这就是为什么线程池对 IO 密集型任务有效的原因：
    #   等待 IO 的线程释放 GIL → 其他线程可以继续执行 Python 代码
    #
    # GIL（Global Interpreter Lock）说明：
    #   - CPython 解释器有一个全局锁，确保同一时刻只有一个线程执行
    #     Python 字节码（严格来说是每个字节码指令都会检查 GIL）
    #   - CPU 密集型任务（纯计算）：多线程是负优化，因为线程间竞争 GIL
    #     反而增加了上下文切换开销
    #   - IO 密集型任务：线程在系统调用时释放 GIL → 真正的并行等待
    #     并发耗时 ≈ 最慢任务耗时（理想情况）
    #   - 要突破 GIL 做 CPU 并行：必须使用 multiprocessing 或 C 扩展
    time.sleep(delay)     # 模拟IO — GIL在这里被释放
    return f"{url} 完成 ({delay:.1f}s)"

urls = [f"page_{i}.html" for i in range(5)]

# ── 串行执行（对比组） ──
# 串行时，每个 download() 阻塞在 time.sleep() 上，CPU 完全空闲等待。
# 总耗时 = sum(每个任务的 delay) ≈ 5 * avg_delay
print("=== 串行 ===")
start = time.time()
for url in urls:
    print(" ", download(url))
print(f"串行耗时: {time.time()-start:.2f}s\n")

# ── 线程池并发 ──
# ThreadPoolExecutor 工作原理：
#   1. max_workers=5：创建一个包含 5 个工作线程的线程池
#   2. executor.submit(download, url)：将任务提交到任务队列
#      返回一个 Future 对象，代表异步执行的结果
#   3. 线程池中的空闲线程从队列中取出任务执行
#   4. 当线程执行到 time.sleep() 时释放 GIL → 其他线程可以执行
#   5. as_completed(futures)：迭代器，每当任意 Future 完成时，
#      立即 yield 该 Future（不按提交顺序返回，按完成顺序）
#   6. future.result()：获取任务返回值，如果任务抛异常则在此处抛出
print("=== 并发 ===")
start = time.time()
with ThreadPoolExecutor(max_workers=5) as executor:
    # executor.submit(download, url) 将任务分发给线程池：
    # 内部有一个 Call Queue（线程安全队列），n 个工作线程从队列中
    # 取出任务并执行。当工作线程空闲时它会从队列取下一个任务；
    # 如果所有线程都忙，提交的任务在队列中等待。
    futures = {executor.submit(download, url): url for url in urls}

    # as_completed 返回一个生成器，每当有 Future 完成时 yield 它。
    # 内部实现：as_completed 用一个回调集合监听每个 Future 的完成事件，
    # 使用 condition variable 实现生产者-消费者模式。
    for future in as_completed(futures):
        print(" ", future.result())
print(f"并发耗时: {time.time()-start:.2f}s")

# 并发耗时 ≈ 最慢任务的时间，远小于串行的各任务时间之和

### 选型速查

| 场景 | 工具 | 理由 |
|------|------|------|
| 下载几十个网页 | `ThreadPoolExecutor` | 简单够用 |
| 上千并发连接 | `asyncio` | 协程开销极小 |
| CPU 密集计算 | `ProcessPoolExecutor` | 绕过 GIL |

---

## 🎯 本课总结

| 特性 | 关键 | 一句话 |
|------|------|--------|
| 生成器 | `yield` | 按需生产，省内存 |
| 装饰器 | `@` | 不改源码，包装功能 |
| 闭包 | 函数+外部变量 | 装饰器的基石 |
| 上下文管理器 | `with` + `yield` | 自动管理资源 |
| 线程池 | `ThreadPoolExecutor` | IO 并发首选 |

### 实战场景 vs 过度设计

| 特性 | 真正的战场 | 杀鸡用牛刀（别这么干） |
|------|-----------|----------------------|
| **生成器** | GB 级日志逐行处理、无限数据流、深度递归转惰性迭代 | 内存几百个元素 → 列表推导式就够了 |
| **装饰器** | 日志/计时/权限/缓存/事务，横切关注点统一管理 | 只给一个函数加功能 → 直接改函数本身更简单 |
| **闭包** | 函数工厂（日志等级、回调上下文）、{! 保持状态 !} | 直接传参数能解决的问题 → 别硬套闭包 |
| **上下文管理器** | 数据库连接/锁/事务的自动清理与回滚 | `with open()` 是必须的，但不用自己写 `@contextmanager` |
| **线程/进程/协程** | 见上方并发选择指南 | 总共 10 条数据 → 串行完事，并发徒增复杂度 |

**最重要的原则：** 先用最简单的方式写对，再用高级特性优化出性能或可维护性。不要为了炫技而用高级特性。

---

## 🧪 练习：文件搜索器（生成器版）

In [ ]:
from pathlib import Path

# ================================================================
# 文件搜索器（生成器版）
# ================================================================
#
# 生成器版本的文件搜索：找到一个匹配的文件就 yield 一个。
# 对比列表版本：先扫描全部文件得到完整列表，再返回给调用者。
#
# 生成器优势：
#   假设有 10 万个文件，第 100 个就匹配了：
#   - 列表版：必须等全部 10 万个文件的扫描完成后才能返回结果
#   - 生成器版：扫描到第 100 个时立即 yield，调用者立刻处理

def search_files(directory, keyword):
    """
    生成器函数：在 directory 中递归搜索包含 keyword 的文件。
    每找到一个匹配文件就 yield 一个结果，然后暂停，等待下一次迭代。
    调用者通过 for 循环依次获取每个结果。
    """

    # Path(directory).rglob("*") 递归遍历所有文件和目录。
    # rglob 本身返回一个生成器——所以整个搜索过程是"逐层惰性"的：
    # 外层生成器每次迭代时才从内层 rglob 生成器中取下一个路径。
    for filepath in Path(directory).rglob("*"):
        if filepath.is_file():
            try:
                # read_text() 是 IO 操作。如果文件很大（如 1GB），
                # 列表版必须将内容读入内存后再判断是否包含关键字。
                # 但生成器在 yield 后函数的局部变量（content）仍有引用，
                # 直到下次 next() 调用前，内存不会被释放。
                # 实际使用中可配合 readline() 逐行读取以进一步省内存。
                content = filepath.read_text(encoding="utf-8")
                if keyword.lower() in content.lower():
                    # yield 生成器函数暂停，将 filepath.absolute() 返回给调用者。
                    # 帧状态（filepath、content、keyword）全部保留。
                    yield filepath.absolute()
            except (UnicodeDecodeError, PermissionError):
                pass

# 搜索包含 "import" 的 Python 文件
print("包含 'import' 的 Python 文件：")
count = 0
for f in search_files(".", "import"):
    # for 循环的每次迭代都会触发一次 next(f)：
    #   - 生成器从上次 yield 处恢复，继续搜索。
    #   - 找到下一个匹配 → 再次 yield → 回到 for 循环体。
    #   - 当 rglob 遍历完成而再无匹配时，生成器函数自然结束，
    #     for 循环捕获 StopIteration 后退出。
    if f.suffix == ".py":
        print(f"  {f}")
        count += 1
print(f"共 {count} 个")

# 生成器优势：找到就显示，不需等所有文件扫描完
# 10万个文件：列表先卡很久，生成器立刻出结果

---

> **恭喜！** 你已完成全部 7 课。从变量、循环、函数一路走到生成器、装饰器、并发——你已经掌握了 Python 的核心武器库。编程不是"学完"的，是"练会"的。
> 接下来：亲手写每课练习 → 做自己感兴趣的项目 → 遇到问题回头翻 notebook。
>
> 这些高级特性就像工具箱里的电动工具：日常小问题用手动工具（基础语法），遇到大项目时再上电动工具（高级特性）。知道它们的存在、理解它们的原理，你写代码时就有更多的选择。
>
> 路线规划：[Python 学习路线规划](../python_learning_plan.md)